# Data Challenge : Lynred data

---

## Imports

In [17]:
# --- Standard Library ---
import os
import glob
import csv

# --- Math & Image Processing ---
import numpy as np
import cv2
from skimage import io
from scipy.ndimage import convolve1d, gaussian_filter1d
from scipy.signal import find_peaks

# --- Parallelization ---
from joblib import Parallel, delayed

# --- Visualization ---
import matplotlib.pyplot as plt

# --- Machine Learning ---
import xgboost as xgb
from scipy.ndimage import uniform_filter1d

# 
from collections import defaultdict

---

## Load Image

In [18]:
def load_img(path):
    """
    Loads an image from the given file path.

    Args:
        path (str): Full path to the image file.

    Returns:
        np.ndarray: The image as a numpy array.
    """
    # Simply read and return the image
    return io.imread(path)

---

## Build all the data set paths

In [19]:
def load_dataset(folder='train', high_dyn=True):
    """
    Builds a dictionary of image paths grouped by type, sequence, and dynamics.

    Args:
        folder (str): Target directory name.
        high_dyn (bool): Whether to include 'high_dyn' in the search.

    Returns:
        tuple: (Dataset dictionary, Flat list of all image paths)
    """
    cam_types = ['HD', 'SXGA', 'VGA']
    seqs = ['sequence_1', 'sequence_2', 'sequence_3']
    
    # Define dynamics based on the high_dyn flag
    dyns = [
        'low dyn', 
        'low dyn with columns 1', 
        'low dyn with columns 2', 
        'low dyn with columns 3'
    ]
    if high_dyn:
        dyns.insert(0, 'high_dyn')

    data_dict = {}
    all_paths = []

    # Build the dictionary and flat list
    for t in cam_types:
        data_dict[t] = {}
        for seq in seqs:
            data_dict[t][seq] = {}
            for dyn in dyns:
                # Grab all PNG files in the specific folder
                pattern = f"{folder}/{t}/{seq}/{dyn}/*.png"
                paths = glob.glob(pattern)
                
                data_dict[t][seq][dyn] = paths
                all_paths.extend(paths)

    return data_dict, all_paths

--- 

## Correction

In [20]:
def make_mask(shape, defects):
    """
    Creates a 2D boolean mask for defective pixels based on coordinates.

    Args:
        shape (tuple): The (height, width) of the target image.
        defects (dict): Keys are x-coords, values are dicts with 'start' and 'stop' y-coords.
            
    Returns:
        np.ndarray: 2D boolean array (True = defect).
    """
    h, w = shape
    mask = np.zeros(shape, dtype=bool)
    
    if not defects:
        return mask

    for x, intervals in defects.items():
        if not (0 <= x < w):
            continue
        
        # 'intervals' est maintenant directement ta liste de tuples [(y_start, y_end), ...]
        # Plus besoin de .get('start') ou .get('stop') !
        for y0, y1 in intervals:
            y_start = max(0, int(y0))
            y_stop = min(h, int(y1))
            mask[y_start:y_stop, x] = True
            
            
    return mask

In [21]:
def fix_stripes(img, defects):
    """
    Corrects column defects using frequency separation and 1D interpolation.
    Non-defective pixels are preserved completely.

    Args:
        img (np.ndarray): 2D input image.
        defects (dict): Defect coordinates dictionary.
            
    Returns:
        np.ndarray: Corrected image (16-bit uint).
    """
    out_img = np.copy(img)
    h, w = out_img.shape
    
    # 1. Generate the exact 2D mask
    mask = make_mask(out_img.shape, defects)
                    
    if not np.any(mask):
        return img.astype(np.uint16)

    img_float = img.astype(np.float32)

    # 2. Vertical Frequency Separation
    # Isolate low frequencies (smooth background) and high frequencies (details)
    low_freq = gaussian_filter1d(img_float, sigma=11.0, axis=0)
    high_freq = img_float - low_freq 

    low_fixed = np.copy(low_freq)
    
    # 3. Horizontal Interpolation on Low Frequencies
    for y in range(h):
        row_mask = mask[y, :]
        if not np.any(row_mask):
            continue
            
        clean_idx = np.where(~row_mask)[0]
        err_idx = np.where(row_mask)[0]
        
        # Interpolate missing low-frequency pixels using clean neighbors
        if len(clean_idx) > 1:
            interp_vals = np.interp(err_idx, clean_idx, low_freq[y, clean_idx])
            low_fixed[y, err_idx] = interp_vals

    # 4. Recombine and format output
    final_float = low_fixed + high_freq
    fixed_16b = np.clip(np.round(final_float), 0, 65535).astype(np.uint16)
    
    # 5. Apply corrections only to defective areas
    out_img[mask] = fixed_16b[mask]
    
    return out_img.astype(np.uint16)

--- 

## Detection

In [22]:
def extract_image_features_vectorized(img_prev, img_curr, img_next):
    # 1. Conversion en float64 pour toute l'image d'un coup
    img_prev = img_prev.astype(np.float64)
    img_curr = img_curr.astype(np.float64)
    img_next = img_next.astype(np.float64)
    
    height, width = img_curr.shape[:2]
    
    # DÉTECTION DE LA RÉSOLUTION (Pour la fenêtre du LT_moy)
    if height < 700: 
        taille_fenetre = 18
    elif height < 1050: 
        taille_fenetre = 31
    else: 
        taille_fenetre = 24

    # --- Statistiques Globales ---
    col_mean = np.mean(img_curr, axis=0) # Vecteur avec la moyenne de chaque colonne
    col_energy = np.sum(img_curr ** 2, axis=0) / height
    
    # =========================================================================
    # 1. LT_moy (Fenêtre glissante ultra-rapide)
    # =========================================================================
    # uniform_filter1d fait la tendance locale pour TOUTES les colonnes instantanément
    tendance_locale = uniform_filter1d(col_mean, size=taille_fenetre, mode='reflect')
    
    # Calcul de l'écart-type local via la variance : V(X) = E(X^2) - E(X)^2
    mean_sq = np.mean(img_curr ** 2, axis=0)
    tendance_sq = uniform_filter1d(mean_sq, size=taille_fenetre, mode='reflect')
    std_locale = np.sqrt(np.maximum(tendance_sq - tendance_locale**2, 0))
    
    ecart_lt_moy = np.abs(col_mean - tendance_locale)
    ratio_lt_moy = ecart_lt_moy / (std_locale + 1e-5)
    
    # =========================================================================
    # 2. Spatial & Temporel
    # =========================================================================
    # Décalage du vecteur pour comparer avec la colonne de gauche et de droite
    left_neighbor = np.roll(col_mean, 1)
    right_neighbor = np.roll(col_mean, -1)
    neighbor_mean = (left_neighbor + right_neighbor) / 2.0
    
    # Correction des extrêmes (bords de l'image)
    neighbor_mean[0] = col_mean[1]
    neighbor_mean[-1] = col_mean[-2]
    
    spatial_diff = np.abs(col_mean - neighbor_mean)
    
    # Contexte temporel
    mean_prev = np.mean(img_prev, axis=0)
    mean_next = np.mean(img_next, axis=0)
    diff_temp_absolue = np.abs(col_mean - mean_prev)
    scintillement_temporel = np.abs(col_mean - ((mean_prev + mean_next) / 2.0))

    # --- Assemblage Final (Seulement nos 7 super-features !) ---
    features_matrix = np.column_stack((
        ecart_lt_moy, ratio_lt_moy,
        spatial_diff, diff_temp_absolue, scintillement_temporel,
        col_mean, col_energy
    ))
    
    return features_matrix

In [23]:
def clf_detect(img_prev, img_curr, img_next, camera_type='VGA'):
    """
    Predict if a column is defect or not using the saved XGBoost baseline.
    Returns the list of defective column indices (x).
    """
    # 1. Charger le modèle (Attention aux guillemets et au f-string)
    clf = xgb.XGBClassifier()
    # On met camera_type en minuscule si tu as sauvegardé sous "xgboost_vga_baseline.json"
    chemin_modele = f'models/xgboost_{camera_type.lower()}_baseline.json'
    clf.load_model(chemin_modele)
    
    # 2. Extraire les features de toute l'image (Tes 7 super-features vectorisées)
    # /!\ Assure-toi que ta fonction extract_image_features_vectorized est bien chargée
    X_features = extract_image_features_vectorized(img_prev, img_curr, img_next)
    
    # 3. Faire la prédiction sur toutes les colonnes d'un coup
    # predictions sera un tableau de 0 (sain) et de 1 (défaut)
    predictions = clf.predict(X_features)
    
    # 4. Trouver les index (x) où la prédiction vaut 1
    colonnes_defectueuses = np.where(predictions == 1)[0]
    
    # On retourne une liste Python classique
    return colonnes_defectueuses.tolist()

### Version alternative avec local threshold pour tester si c'est mieux que random forest

In [24]:
def local_threshold(image, taille_fenetre=50, facteur_std=1.0):

    # height = image.shape[0]  plus besoin ici !
    
    metrique_ = np.mean(image, axis=0)
    label = 'Moyenne'
    
    filtre = np.ones(taille_fenetre) / taille_fenetre
    tendance_locale = convolve1d(metrique_, filtre, mode='reflect')
    
    # Marge de tolérance
    marge_tolerance = facteur_std * np.std(metrique_)
    
    # Le seuil n'est plus un simple nombre, c'est un tableau de la même taille que l'image !
    threshold_haut = tendance_locale + marge_tolerance
    threshold_bas = tendance_locale - marge_tolerance
    
    # Détection bilatérale avec l'écart absolu (np.abs)
    defect_columns = np.where(np.abs(metrique_ - tendance_locale) > marge_tolerance)[0]

    # On renvoie juste une liste de 'int' pour chaque cols 
    return [int(col) for col in defect_columns]

In [25]:

# =========================================================
# 1. LA FONCTION DE FUSION 
# =========================================================
def fusionner_segments(segments_etendus, tolerance=15):
    if not segments_etendus:
        return {}

    dict_intermediaire = defaultdict(list)
    for segment in segments_etendus:
        x = segment[0]
        y_start = segment[1]
        y_end = segment[2]
        dict_intermediaire[x].append((y_start, y_end))

    dict_final = {}
    for x, liste_segments in dict_intermediaire.items():
        if len(liste_segments) <= 1:
            dict_final[x] = liste_segments
            continue
            
        segments_tries = sorted(liste_segments, key=lambda coord: coord[0])
        segments_fusionnes = [segments_tries[0]]
        
        for segment_actuel in segments_tries[1:]:
            dernier_segment_valide = segments_fusionnes[-1]
            debut_actuel, fin_actuelle = segment_actuel
            debut_dernier, fin_derniere = dernier_segment_valide
            
            if debut_actuel <= (fin_derniere + tolerance):
                nouvelle_fin = max(fin_derniere, fin_actuelle)
                segments_fusionnes[-1] = (debut_dernier, nouvelle_fin)
            else:
                segments_fusionnes.append(segment_actuel)
                
        dict_final[x] = segments_fusionnes

    return dict_final

# =========================================================
# 2. TON ALGORITHME (Le Region Growing)
# =========================================================
def detect_defects(img, colonnes_rf_array, multiplicateur_rupture=3.5, taille_lissage=11):
    height = img.shape[0]
    segments_etendus = []
    
    filtre = np.ones(taille_lissage) / taille_lissage
    colonnes_rf = np.atleast_1d(colonnes_rf_array)

    for x in colonnes_rf:
        x = int(x)
        colonne = img[:, x].astype(np.float32)
        
        colonne_lissee = convolve1d(colonne, filtre, mode='reflect')
        mediane_col = np.median(colonne_lissee)
        signal_anomalie = np.abs(colonne_lissee - mediane_col)
        
        seuil_bruit = 2 * np.std(signal_anomalie) 
        graines, _ = find_peaks(signal_anomalie, height=seuil_bruit, distance=10)
        
        if len(graines) == 0:
            graines = [np.argmax(signal_anomalie)]

        diffs = np.abs(np.diff(colonne))
        bruit_normal = np.median(diffs)
        ecart_sauts = np.std(diffs)
        seuil_rupture = bruit_normal + (multiplicateur_rupture * ecart_sauts)

        murs = np.where(diffs >= seuil_rupture)[0]

        for y_seed in graines:
            murs_haut = murs[murs < y_seed]
            y_start = int(murs_haut[-1] + 1) if len(murs_haut) > 0 else 0

            murs_bas = murs[murs >= y_seed]
            y_end = int(murs_bas[0]) if len(murs_bas) > 0 else height - 1
                
            # C'est ce format [x, y_start, y_end] que la nouvelle fonction de fusion attend !
            segments_etendus.append([x, y_start, y_end])

    return fusionner_segments(segments_etendus)

In [31]:
data_dict, _ = load_dataset(folder='train', high_dyn=False)
print(len(data_dict))

3


---

## Run

In [32]:
# --- Configuration by Image Type ---
PARAMS_LT = {
    'VGA':  {'taille_fenetre': 16, 'facteur_std': 1.918},
    'HD':   {'taille_fenetre': 24, 'facteur_std': 1.26},
    'SXGA': {'taille_fenetre': 31, 'facteur_std': 1.74}
}

PARAMETRES_REGION_GROWING = { 
'VGA': {'multiplicateur_rupture': 5.9,  'taille_lissage': 21},  
'SXGA': {'multiplicateur_rupture': 6,  'taille_lissage': 15},  
'HD': {'multiplicateur_rupture': 3.84,  'taille_lissage': 3} 
}

def process_img(cam_type, seq, dyn, img_path):
    """
    Processes a single image: loads, detects defects, fixes them, and saves the result.

    Args:
        cam_type (str): 'VGA', 'SXGA', or 'HD'.
        seq (str): Sequence name.
        dyn (str): Dynamics type.
        img_path (str): Full path to the input image.
    """
    filename = os.path.basename(img_path)
    folder = os.path.dirname(img_path)
    
    # Setup results directory
    res_folder = os.path.join(folder, "results")
    os.makedirs(res_folder, exist_ok=True)
    save_path = os.path.join(res_folder, filename)
    
    # 1. Load Image
    img = load_img(img_path)
    
    # 2. Get specific parameters (Avec les bonnes clés par défaut !)
    p_rg = PARAMETRES_REGION_GROWING.get(cam_type, {'multiplicateur_rupture': 3.5, 'taille_lissage': 11}) 
    p_seed = PARAMS_LT.get(cam_type, {'taille_fenetre': 50, 'facteur_std': 1.0})

    # 3. Detect Defects
    colonnes_suspectes = local_threshold(img, **p_seed)
    
    defects = detect_defects(
        img, 
        colonnes_rf_array=colonnes_suspectes,
        multiplicateur_rupture=p_rg['multiplicateur_rupture'], 
        taille_lissage=p_rg['taille_lissage']                  
    )
    
    # 4. Correct Image if defects are found
    if not defects:
        fixed_img = np.copy(img)
    else:
        fixed_img = fix_stripes(img, defects)

    # 5. Format and Save
    out_img = np.clip(fixed_img, 0, 65535).astype(np.uint16)
    
    # Double write to prevent OS caching/disk write errors
    cv2.imwrite(save_path, out_img)
    success = cv2.imwrite(save_path, out_img)
    
    if not success:
        print(f"ERROR: OpenCV failed to save {save_path}")
        return None

# ==========================================
# --- MAIN RUN SCRIPT ---
# ==========================================

# 1. Load Dataset
data_dict, _ = load_dataset(folder='train', high_dyn=False)

# 2. Prepare Task List
tasks = []
for t, seqs in data_dict.items():
    for seq, dyns in seqs.items():
        for dyn, paths in dyns.items():
            if dyn == "low dyn":
                continue 
            for path in paths:
                tasks.append((t, seq, dyn, path))

print(f"Launching Joblib for {len(tasks)} images...")

# 3. Parallel Execution
raw_results = Parallel(n_jobs=-1)(
    delayed(process_img)(*task) for task in tasks
)

TypeError: 'dict_items' object is not subscriptable

In [34]:
# --- Configuration by Image Type ---
PARAMS_LT = {
    'VGA':  {'taille_fenetre': 16, 'facteur_std': 1.918},
    'HD':   {'taille_fenetre': 24, 'facteur_std': 1.26},
    'SXGA': {'taille_fenetre': 31, 'facteur_std': 1.74}
}

PARAMETRES_REGION_GROWING = { 
'VGA': {'multiplicateur_rupture': 5.9,  'taille_lissage': 21},  
'SXGA': {'multiplicateur_rupture': 6,  'taille_lissage': 15},  
'HD': {'multiplicateur_rupture': 3.84,  'taille_lissage': 3} 
}

def process_img(cam_type, seq, dyn, img_path):
    """
    Processes a single image: loads, detects defects, fixes them, and saves the result.

    Args:
        cam_type (str): 'VGA', 'SXGA', or 'HD'.
        seq (str): Sequence name.
        dyn (str): Dynamics type.
        img_path (str): Full path to the input image.
    """
    filename = os.path.basename(img_path)
    folder = os.path.dirname(img_path)
    
    # Setup results directory
    res_folder = os.path.join(folder, "results")
    os.makedirs(res_folder, exist_ok=True)
    save_path = os.path.join(res_folder, filename)
    
    # 1. Load Image
    img = load_img(img_path)
    
    # 2. Get specific parameters (Avec les bonnes clés par défaut !)
    p_rg = PARAMETRES_REGION_GROWING.get(cam_type, {'multiplicateur_rupture': 3.5, 'taille_lissage': 11}) 
    p_seed = PARAMS_LT.get(cam_type, {'taille_fenetre': 50, 'facteur_std': 1.0})

    # 3. Detect Defects
    colonnes_suspectes = local_threshold(img, **p_seed)
    
    defects = detect_defects(
        img, 
        colonnes_rf_array=colonnes_suspectes,
        multiplicateur_rupture=p_rg['multiplicateur_rupture'], 
        taille_lissage=p_rg['taille_lissage']                  
    )
    
    # 4. Correct Image if defects are found
    if not defects:
        fixed_img = np.copy(img)
    else:
        fixed_img = fix_stripes(img, defects)

    # 5. Format and Save
    out_img = np.clip(fixed_img, 0, 65535).astype(np.uint16)
    
    # Double write to prevent OS caching/disk write errors
    cv2.imwrite(save_path, out_img)
    success = cv2.imwrite(save_path, out_img)
    
    if not success:
        print(f"ERROR: OpenCV failed to save {save_path}")
        return None

# ==========================================
# --- MAIN RUN SCRIPT (AVEC AUDIT) ---
# ==========================================

print("📥 DÉBUT DU CHARGEMENT DU DATASET...")

# 1. Load Dataset
data_dict, _ = load_dataset(folder='train', high_dyn=False)

# 2. Prepare Task List et Audit
tasks = []
total_images_trouvees = 0

print("\n🔍 DÉTAIL DES DOSSIERS TROUVÉS :")
print("-" * 50)

for t, seqs in data_dict.items():
    for seq, dyns in seqs.items():
        for dyn, paths in dyns.items():
            
            nb_images = len(paths)
            total_images_trouvees += nb_images
            
            if dyn == "low dyn":
                print(f"⚠️  Ignoré   : {t:4} | {seq:10} | {dyn:25} -> {nb_images:4} images (Skipped)")
                continue 
                
            print(f"✅ Conservé : {t:4} | {seq:10} | {dyn:25} -> {nb_images:4} images")
            
            for path in paths:
                tasks.append((t, seq, dyn, path))

print("-" * 50)
print(f"Total absolu d'images sur le disque dur : {total_images_trouvees}")
print(f"Total d'images à traiter par Joblib     : {len(tasks)}")
print("-" * 50 + "\n")

if len(tasks) == 0:
    print("❌ ERREUR CRITIQUE : Aucune image à traiter. Joblib ne se lance pas.")
    print("-> Vérifie que le dossier 'train' est bien au même endroit que ton Notebook.")
    print("-> Vérifie que les noms des dossiers (espaces, majuscules) correspondent exactement à 'low dyn with columns 1', etc.")
else:
    print(f"🚀 Launching Joblib for {len(tasks)} images...")
    # 3. Parallel Execution
    raw_results = Parallel(n_jobs=-1)(
        delayed(process_img)(*task) for task in tasks
    )
    print("✅ Traitement Joblib terminé !")

📥 DÉBUT DU CHARGEMENT DU DATASET...

🔍 DÉTAIL DES DOSSIERS TROUVÉS :
--------------------------------------------------
⚠️  Ignoré   : HD   | sequence_1 | low dyn                   ->  300 images (Skipped)
✅ Conservé : HD   | sequence_1 | low dyn with columns 1    ->  300 images
✅ Conservé : HD   | sequence_1 | low dyn with columns 2    ->  300 images
✅ Conservé : HD   | sequence_1 | low dyn with columns 3    ->  300 images
⚠️  Ignoré   : HD   | sequence_2 | low dyn                   ->  300 images (Skipped)
✅ Conservé : HD   | sequence_2 | low dyn with columns 1    ->  300 images
✅ Conservé : HD   | sequence_2 | low dyn with columns 2    ->  300 images
✅ Conservé : HD   | sequence_2 | low dyn with columns 3    ->  300 images
⚠️  Ignoré   : HD   | sequence_3 | low dyn                   ->  300 images (Skipped)
✅ Conservé : HD   | sequence_3 | low dyn with columns 1    ->  300 images
✅ Conservé : HD   | sequence_3 | low dyn with columns 2    ->  300 images
✅ Conservé : HD   | sequence_3

---

## Evaluation

### ANcienne version qui me donne rien

In [42]:
import os
import csv
import numpy as np
import importlib
import metrics

# Cette ligne magique force ton Notebook à lire la correction que tu viens de faire !
importlib.reload(metrics) 
from metrics import evaluate_sequence
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- Configuration ---
root_path = "train" 
sensors   = ["HD", "SXGA", "VGA"]
csv_path  = "results.csv"

# 1. Task Construction
tasks = []
for sensor in sensors:
    sensor_path = os.path.join(root_path, sensor)
    if os.path.exists(sensor_path):
        for entry in os.scandir(sensor_path):
            if entry.is_dir():
                for sim in (1, 2, 3):
                    tasks.append((sensor, entry.path, sim))

print(f"Starting evaluation for {len(tasks)} tasks...\n" + "-"*60)

results = []
final_scores = []

# 2. Parallel Execution with REAL-TIME display
with ProcessPoolExecutor() as pool:
    # Submit all tasks to the pool
    futures = {pool.submit(evaluate_sequence, task): task for task in tasks}    
    for future in as_completed(futures):
        res = future.result()
        if res is not None:
            results.append(res)
            final_scores.append(float(res[13]))
            print(res[-1])

print("-" * 60)

# 3. CSV Writing once everything is finished
if results:
    with open(csv_path, mode="w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow([
            "sensor", "sequence", "def_path", "TP", "FP", "FN",
            "precision", "recall", "F1",
            "RMSE_def", "RMSE_ok", "RMSE_def_norm",
            "RMSE_ok_norm", "final_score"
        ])
        for r in results:
            writer.writerow(r[:-1])

# 4. Final Score Display
if final_scores:
    mean_score = np.mean(final_scores)
    print("\nEvaluation completed! CSV file generated.")
    print("=== MEAN FINAL SCORE ACROSS ALL SEQUENCES ===")
    print(f"Mean score = {mean_score:.6f}")
else:
    print("\nNo results were produced.")

Starting evaluation for 27 tasks...
------------------------------------------------------------
------------------------------------------------------------

No results were produced.


In [43]:
from metrics import evaluate_sequence
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- Configuration ---
root_path = "train" 
sensors   = ["HD", "SXGA", "VGA"]
csv_path  = "results.csv"

# 1. Task Construction
tasks = []
for sensor in sensors:
    sensor_path = os.path.join(root_path, sensor)
    if os.path.exists(sensor_path):
        for entry in os.scandir(sensor_path):
            if entry.is_dir():
                for sim in (1, 2, 3):
                    tasks.append((sensor, entry.path, sim))

print(f"Starting evaluation for {len(tasks)} tasks...\n" + "-"*60)

results = []
final_scores = []

# 2. Parallel Execution with REAL-TIME display
with ProcessPoolExecutor() as pool:
    # Submit all tasks to the pool
    futures = {pool.submit(evaluate_sequence, task): task for task in tasks}    
    for future in as_completed(futures):
        res = future.result()
        if res is not None:
            results.append(res)
            final_scores.append(float(res[13]))
            print(res[-1])

print("-" * 60)

# 3. CSV Writing once everything is finished
if results:
    with open(csv_path, mode="w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow([
            "sensor", "sequence", "def_path", "TP", "FP", "FN",
            "precision", "recall", "F1",
            "RMSE_def", "RMSE_ok", "RMSE_def_norm",
            "RMSE_ok_norm", "final_score"
        ])
        for r in results:
            writer.writerow(r[:-1])

# 4. Final Score Display
if final_scores:
    mean_score = np.mean(final_scores)
    print("\nEvaluation completed! CSV file generated.")
    print("=== MEAN FINAL SCORE ACROSS ALL SEQUENCES ===")
    print(f"Mean score = {mean_score:.6f}")
else:
    print("\nNo results were produced.")

Starting evaluation for 27 tasks...
------------------------------------------------------------
[HD/sequence_1 1] TP=0.006 FP=0.011 FN=0.002 Prec=0.371 Rec=0.790 F1=0.505 RMSE_def=7.80 RMSE_ok=1.79 RMSE_def_norm=0.81 RMSE_ok_norm=0.96 Score final : 0.7525
[HD/sequence_1 2] TP=0.003 FP=0.011 FN=0.003 Prec=0.208 Rec=0.487 F1=0.292 RMSE_def=6.49 RMSE_ok=1.85 RMSE_def_norm=0.84 RMSE_ok_norm=0.95 Score final : 0.6904
[HD/sequence_2 3] TP=0.005 FP=0.001 FN=0.008 Prec=0.859 Rec=0.362 F1=0.509 RMSE_def=11.32 RMSE_ok=0.16 RMSE_def_norm=0.72 RMSE_ok_norm=1.00 Score final : 0.7384
[HD/sequence_2 2] TP=0.004 FP=0.000 FN=0.008 Prec=0.999 Rec=0.319 F1=0.483 RMSE_def=13.31 RMSE_ok=0.01 RMSE_def_norm=0.67 RMSE_ok_norm=1.00 Score final : 0.7144
[HD/sequence_2 1] TP=0.009 FP=0.002 FN=0.003 Prec=0.821 Rec=0.761 F1=0.790 RMSE_def=7.17 RMSE_ok=0.28 RMSE_def_norm=0.82 RMSE_ok_norm=0.99 Score final : 0.8673
[HD/sequence_3 1] TP=0.006 FP=0.003 FN=0.002 Prec=0.679 Rec=0.741 F1=0.709 RMSE_def=8.96 RMSE_ok=0.51